# Segmenting Real Customers with RFM

**Week 4 · Lesson 12** &nbsp;·&nbsp; GBUS 564: Predictive Analytics &nbsp;·&nbsp; Fall 2026

<a target="_blank" href="https://colab.research.google.com/github/avorybryant/umw_gbus564_fall26/blob/main/notebooks/week4/lectures/12_segmenting_customers_rfm.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

*Adapted from ISLP Ch. 12.5 Lab where relevant.*

## Setup

Same boilerplate as every notebook this term — see Week 1 Lesson 8 if this is your
first time seeing it.

In [ ]:
# Environment setup — identical across every notebook in the course.
# Installs the course package (dataset loaders + helpers) from the public
# course repo; Colab already has numpy/pandas/scikit-learn, so it's quick.
%pip install -q matplotlib git+https://github.com/avorybryant/umw_gbus564_fall26.git

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import set_config

set_config(display="diagram")
plt.rcParams["figure.figsize"] = (8, 5)

RANDOM_SEED = 564  # GBUS 564 — reuse this seed everywhere for reproducibility
rng = np.random.default_rng(RANDOM_SEED)

## 1. Business Problem

**Business question:** which customers should get a loyalty reward, which need a
win-back offer before they're gone for good, and which are still too new to read —
and can that split come from purchase behavior alone, with no survey and no
demographic data?

**Why it matters:** Lessons 9–11 built k-means, learned to choose k, and added
density clustering and embeddings — all on synthetic or toy examples. This notebook is where
clustering meets a real marketing decision: segment the same Online Retail
customers Lesson 8's recommender used, this time aggregated to one row per
customer instead of one row per invoice line, and turn the segments into a
distinct action per group.

## 2. Data

*Load, describe, and sanity-check the data before touching a model.*

Lesson 8 used **invoice lines**; clustering needs one row **per customer**. The
classic recipe for exactly this is **RFM**: for every customer, how **R**ecently
did they last buy, how **F**requently do they buy, and how much do they spend
(**M**onetary)? `load_online_retail()` does the same universal cleaning Lesson 8
relied on (drops guest orders, cancellations, non-positive rows) — the cell below
aggregates that cleaned line-item data up to the customer level.

In [ ]:
from gbus564_coursework.utils.course_data import load_online_retail

df = load_online_retail()
df["TotalPrice"] = df["Quantity"] * df["UnitPrice"]
df.head()

In [ ]:
# Recency is measured relative to the day after the dataset's last transaction —
# there's no "today" in a year-old static file, so this stands in for it.
reference_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)

rfm = df.groupby("CustomerID").agg(
    Recency=("InvoiceDate", lambda s: (reference_date - s.max()).days),
    Frequency=("InvoiceNo", "nunique"),
    Monetary=("TotalPrice", "sum"),
)
print(f"{len(rfm):,} customers")
rfm.describe()

### Data Prep

k-means groups customers by **Euclidean distance** — and Euclidean distance
*squares* every gap, so one huge gap outweighs everything else (Lesson 1's
Spike vs. Scale-up). That makes two fixes necessary before k-means sees this data.

First, **Frequency** and **Monetary** are heavily right-skewed (skew of 12 and 19
respectively — a handful of wholesale accounts buy constantly and spend orders of
magnitude more than a typical customer). A **log transform** turns ratios into
equal steps — $100 → $1,000 becomes the same distance as $1,000 → $10,000 — so
those accounts stop dominating every distance without being dropped. Second,
Lesson 2's rule still applies: **standardize** everything so Recency (measured
in days) doesn't dominate Frequency (a small integer) purely because its numbers
happen to be bigger.

Standardizing alone does *not* fix the first problem — it only shifts and
stretches each column, so an outlier is just as extreme afterward. The
comparison below shows what that does to k-means.

In [ ]:
from sklearn.preprocessing import StandardScaler

rfm_log = np.log1p(rfm)
print("Skew before log1p:")
print(rfm.skew())
print("Skew after log1p:")
print(rfm_log.skew())

scaler = StandardScaler()
X = scaler.fit_transform(rfm_log)

In [ ]:
from sklearn.cluster import KMeans

# Same k-means (k=4) on two versions of the data: standardized only, vs.
# log-transformed then standardized.
for name, data in [("scale only", rfm), ("log + scale", rfm_log)]:
    X_try = StandardScaler().fit_transform(data)
    labels = KMeans(n_clusters=4, n_init=10, random_state=RANDOM_SEED).fit_predict(X_try)
    sizes = pd.Series(labels).value_counts().sort_values().tolist()
    print(f"{name:12s} most extreme Monetary z-score: {X_try[:, 2].max():5.1f}   "
          f"segment sizes: {sizes}")

Without the log, one customer sits about 31 standard deviations out on
Monetary, and k-means spends a whole segment on roughly a dozen wholesale
accounts while lumping most customers into one giant group — "whales vs.
everyone," which marketing can't do much with. With the log, the most extreme
customer is under 5 standard deviations out and the four segments are all
substantial. The wholesale accounts are still there; they just no longer decide
where the segments fall.

Lesson 9 named k-means's round-cluster assumption. Before trusting k-means on
this RFM data, it's worth actually looking — not assuming. With only three
features there's nothing to compress first: the cell below plots every pair of
them, on the same log-transformed, standardized scale k-means will see.

In [ ]:
# Frequency (orders) and Recency (days) are whole numbers, so their small values
# stack into stripes. Spread each count across its own width for the plot only;
# k-means still uses the unspread X.
spread = rfm.astype(float)
for col in ["Recency", "Frequency"]:
    spread[col] += rng.uniform(-0.5, 0.5, len(spread))
X_plot = pd.DataFrame(scaler.transform(np.log1p(spread)), columns=rfm.columns)

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
for ax, (x, y) in zip(axes, [("Recency", "Frequency"), ("Recency", "Monetary"), ("Frequency", "Monetary")]):
    ax.scatter(X_plot[x], X_plot[y], s=6, alpha=0.3, color="#1A73E8")
    ax.set_xlabel(x)
    ax.set_ylabel(y)
fig.suptitle("RFM customers, every pair of features (log, standardized)")
plt.tight_layout()
plt.show()

The RFM cloud is a single connected mass — dense in one corner and fanning out
from there, but no crescents and no separate stripes like Lesson 11's failure
cases. That's a reasonable green light for
k-means here, with one honest caveat carried into Recommend: a single continuous
cloud has no obviously separated groups baked in, so whatever k-means produces
is a **business partition of a continuum**, not a discovery of naturally
occurring tribes. (The plot spreads each whole-number count slightly so that,
say, every two-order customer doesn't stack on one line; that's for display
only.)

## 3. Model

*Baseline first, then k-means.*

There's no supervised baseline here — no label to beat with a mean or
majority-class guess. The nearest equivalent is **k=1**: one segment, i.e. no
segmentation at all, treating every customer identically. Every k above 1 has to
earn its complexity by finding structure that's actually useful to act on.

### Model Selection

Lesson 10 built exactly this recipe: sweep k, check the elbow, check silhouette.
Applying it to real data adds a wrinkle worth watching for.

In [ ]:
from sklearn.metrics import silhouette_score

ks = range(2, 9)
inertias, sil_scores = [], []
for k in ks:
    km_k = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_SEED).fit(X)
    inertias.append(km_k.inertia_)
    sil_scores.append(silhouette_score(X, km_k.labels_))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(list(ks), inertias, "o-", color="#1A73E8")
axes[0].set_xlabel("k")
axes[0].set_ylabel("Inertia")
axes[0].set_title("Elbow")
axes[1].plot(list(ks), sil_scores, "o-", color="#34A853")
axes[1].set_xlabel("k")
axes[1].set_ylabel("Mean silhouette")
axes[1].set_title("Silhouette")
plt.tight_layout()
plt.show()

print(dict(zip(ks, [round(s, 3) for s in sil_scores])))

Silhouette's numeric peak is at **k=2** — but Lesson 10 warned about exactly this
trap: a statistically clean k isn't automatically a *useful* one. Two segments
("engaged" vs. "not") gives marketing almost nothing to act on differently. The
elbow curve keeps bending noticeably through about k=4 before flattening, and
k=4's silhouette (shown below) is still a solid, reasonable score — the business
call this notebook makes is **k=4**: coarse enough to keep every segment large
and interpretable, granular enough to actually target differently.

In [ ]:
from sklearn.metrics import silhouette_samples

# Lesson 10's per-customer view: k=2 (silhouette's peak) vs. k=4 (the business call)
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True)
for ax, k in zip(axes, [2, 4]):
    labels = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_SEED).fit_predict(X)
    s = silhouette_samples(X, labels)  # one score per customer
    y = 0
    for c in range(k):
        vals = np.sort(s[labels == c])
        ax.fill_betweenx(np.arange(y, y + len(vals)), 0, vals, alpha=0.85)  # thickness = segment size
        ax.text(0.02, y + len(vals) / 2, f"{len(vals):,} customers", va="center")
        y += len(vals) + 60  # gap between segments
    ax.axvline(s.mean(), color="#D93025", linestyle="--")  # mean silhouette
    ax.axvline(0, color="gray", linewidth=0.8)
    ax.set_title(f"k = {k}   (mean = {s.mean():.2f})")
    ax.set_xlabel("Silhouette s(i)")
    ax.set_yticks([])
axes[0].set_ylabel("Customers, sorted within each segment")
plt.tight_layout()
plt.show()

Each band is one segment: its thickness is the segment's size, and how far right
it reaches is how well its customers fit. k=2's higher score comes from two big
bands, roughly "engaged" (1,668 customers) vs. "not" (2,670). k=4 scores lower,
but not because a segment broke: all four bands are substantial (721 to 1,589
customers), and only about 2% of customers score below zero. The lower mean is
the cost of cutting one continuous cloud more finely — the Data Prep caveat
again.

What k=2 can't do is separate customers who need opposite actions. Of the 850
customers k=4 puts in its recent-but-new segment (Interpret names it), 599 share
k=2's "not engaged" group with all 1,589 long-lapsed customers — one campaign
where marketing needs two: an onboarding offer and a win-back offer.

In [ ]:
K = 4
kmeans = KMeans(n_clusters=K, n_init=10, random_state=RANDOM_SEED)
rfm["Segment"] = kmeans.fit_predict(X)
print(f"k={K} silhouette: {silhouette_score(X, rfm['Segment']):.3f}")

## 4. Evaluate

*Score against the baseline, not in isolation — and check the split is actually
usable before trusting it.*

In [ ]:
segment_sizes = rfm["Segment"].value_counts().sort_index()
print("Customers per segment:")
print(segment_sizes)
print(f"\nSmallest segment is {segment_sizes.min() / len(rfm):.1%} of all customers — "
      f"large enough for a dedicated campaign, not a rounding error.")

No segment is a sliver of the customer base, and k=4 clears k=1's implicit
"treat everyone the same" baseline by definition — the real test is whether the
segments mean anything, which Interpret takes on next.

## 5. Interpret

*What actually distinguishes these four groups — business-readable, not just
statistically distinct?*

In [ ]:
profile = rfm.groupby("Segment")[["Recency", "Frequency", "Monetary"]].mean().round(1)
profile["n_customers"] = segment_sizes
profile.sort_values("Monetary", ascending=False)

In [ ]:
# Names come from reading the profile table above; the segment numbers stay put
# because k-means is seeded with RANDOM_SEED
segment_names = {1: "Champions", 0: "Core Regulars", 3: "Promising New", 2: "Lapsed / At-Risk"}

# Days, orders, and pounds on one scale: each segment's average as a multiple of
# the average customer's (1.0 = average)
cols = ["Recency", "Frequency", "Monetary"]
vs_average = profile[cols] / rfm[cols].mean()
bar_labels = ["Days since\nlast order", "Orders", "Spend"]

fig, axes = plt.subplots(1, 4, figsize=(14, 4.5), sharey=True)
for ax, (seg, name) in zip(axes, segment_names.items()):
    bars = ax.bar(bar_labels, vs_average.loc[seg], color=["#1A73E8", "#34A853", "#D93025"])
    ax.bar_label(bars, fmt="%.1f×", padding=2)
    ax.axhline(1, color="gray", linestyle="--", linewidth=1)  # the average customer
    ax.set_title(f"{name}\n({segment_sizes[seg]:,} customers)")
axes[0].set_ylabel("× the average customer")
fig.suptitle("Each segment vs. the average customer (dashed line = average)")
plt.tight_layout()
plt.show()

Four readable profiles fall out of the numbers: a **Champions** segment (very
recent, frequent, high spend), a **Core Regulars** segment (moderate on all
three — the steady middle), a **Promising New** segment (recent but only one or
two purchases so far — too little history to read yet), and a **Lapsed/At-Risk**
segment (haven't bought in months, low frequency and spend even when they did).
None of this used a product category, a demographic, or a survey — purely the
shape of each customer's transaction history.

## 6. Recommend

**Recommendation:** four different campaigns, not one blanket email. **Champions**
get a loyalty/VIP perk — protecting the customers already driving
disproportionate revenue costs less than acquiring new ones. **Core Regulars**
get a modest cross-sell nudge to push them toward Champion territory. **Promising
New** customers get an onboarding/second-purchase incentive — there's too little
history yet to know if they'll become Champions or churn. **Lapsed/At-Risk**
customers get a time-limited win-back offer, evaluated against its cost — Lesson
10's actionability question applies directly: a segment marketing can't
profitably act on isn't worth maintaining separately.

The Data Prep assumption check matters here too: this RFM cloud was a single
convex mass, not naturally multi-modal, so these four segments are a **useful
partition of a continuum**, not four naturally occurring tribes — a caveat worth
attaching to the segmentation itself, not just this notebook.

- **RFM** (Recency, Frequency, Monetary) turns raw transactions into one row per
  customer, ready for k-means
- **Log-transform + standardize** before clustering — the same Lesson 2 lesson,
  applied to genuinely skewed real data this time
- **Looking at the data's shape first** — a scatterplot of every feature pair before clustering —
  is what makes trusting k-means's output here a checked decision, not a hope
- **k=4** beat silhouette's raw k=2 peak on business grounds: actionability over
  a marginally higher score, exactly Lesson 10's lesson

**Where this week ends:** Lesson 9 opened with a plain assign/update loop and an
uncomfortable truth — no label to check against. Four lessons later, that same
loop is producing four named, targetable customer segments — judged by whether
marketing can act on them, because on real customers that's the only check
there is.
Week 5 picks the supervised thread back up — CART decision trees, on the same
portfolio of business problems.